In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Đang chuẩn bị Tensor Data...")
# 1. Tải và chuẩn bị dữ liệu
df = pd.read_csv('../data/synthetic_intersectional_data.csv')

# Tạo nhóm giao thoa và mã hóa thành số nguyên (để dùng làm index cho Tensor)
df['inter_group'] = df['gender'] + "_" + df['race']
le = LabelEncoder()
df['group_idx'] = le.fit_transform(df['inter_group'])

features = ['age', 'education_years', 'income']
X = df[features].values
y = df['loan_approved'].values
groups = df['group_idx'].values

# Chuẩn hóa dữ liệu (rất quan trọng cho Neural Networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42)

# Chuyển đổi sang PyTorch Tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
g_train_t = torch.LongTensor(g_train)

X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)
g_test_t = torch.LongTensor(g_test)

# 2. Định nghĩa Mạng Neural Network (MLP)
class FairMLP(nn.Module):
    def __init__(self, input_dim):
        super(FairMLP, self).__init__()
        self.layer1 = nn.Linear(input_dim, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return self.sigmoid(out)

# 3. Định nghĩa Hàm Loss Đa Mục Tiêu (Multi-objective Custom Loss)
def causal_fairness_loss(predictions, targets, groups, lambda_fairness=0.8):
    # Mục tiêu 1: BCE Loss (Độ chính xác - Accuracy)
    bce_loss = nn.BCELoss()(predictions, targets)
    
    # Mục tiêu 2: Tensor-based Intersectional Fairness Loss
    unique_groups = torch.unique(groups)
    group_rates = []
    
    # Tính tỷ lệ duyệt dự đoán trung bình cho từng nhóm bằng các phép toán Tensor
    for g in unique_groups:
        mask = (groups == g)
        if mask.sum() > 0:
            group_rate = predictions[mask].mean()
            group_rates.append(group_rate)
            
    group_rates = torch.stack(group_rates)
    
    # Phạt sự chênh lệch (variance) giữa các nhóm: Variance càng nhỏ, DSC càng gần 1
    fairness_penalty = torch.var(group_rates)
    
    # Tổng hợp Loss
    total_loss = bce_loss + lambda_fairness * fairness_penalty
    return total_loss, bce_loss, fairness_penalty

# 4. Quá trình Huấn luyện (Training Loop)
model = FairMLP(input_dim=3)
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Đang huấn luyện mô hình PyTorch với Multi-objective Optimization...")
epochs = 500
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    predictions = model(X_train_t)
    loss, bce, fair_pen = causal_fairness_loss(predictions, y_train_t, g_train_t, lambda_fairness=2.0)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Total Loss: {loss.item():.4f} | BCE: {bce.item():.4f} | Fair Penalty: {fair_pen.item():.6f}")

# 5. Đánh giá Mô hình
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_preds_class = (test_preds >= 0.5).float()
    
    acc = (test_preds_class == y_test_t).float().mean().item()
    
    # Tính Intersectional DSC
    results_df = pd.DataFrame({
        'y_pred': test_preds_class.squeeze().numpy(),
        'group': g_test_t.numpy()
    })
    approval_rates = results_df.groupby('group')['y_pred'].mean()
    dsc = approval_rates.min() / approval_rates.max()
    
    print("\n✅ KẾT QUẢ MÔ HÌNH TENSOR-BASED FAIRNESS (PYTORCH):")
    print(f"Accuracy: {acc:.4f}")
    print(f"Intersectional DSC: {dsc:.4f}")

Đang chuẩn bị Tensor Data...
Đang huấn luyện mô hình PyTorch với Multi-objective Optimization...
Epoch 100/500 | Total Loss: 0.2916 | BCE: 0.2910 | Fair Penalty: 0.000341
Epoch 200/500 | Total Loss: 0.2850 | BCE: 0.2837 | Fair Penalty: 0.000631
Epoch 300/500 | Total Loss: 0.2840 | BCE: 0.2827 | Fair Penalty: 0.000631
Epoch 400/500 | Total Loss: 0.2839 | BCE: 0.2826 | Fair Penalty: 0.000634
Epoch 500/500 | Total Loss: 0.2838 | BCE: 0.2826 | Fair Penalty: 0.000635

✅ KẾT QUẢ MÔ HÌNH TENSOR-BASED FAIRNESS (PYTORCH):
Accuracy: 0.9007
Intersectional DSC: 0.9994
